In [1]:
import torch as t
from transformer_lens import HookedTransformer, HookedTransformerConfig
from rl_agents.agents.deep_q_network.pytorch import DQNAgent
from rl_agents.agents.common.factory import load_agent, load_environment
import plotly.express as px
from circuitsvis.attention import attention_patterns

/Users/vasudharajain/workspace/rl-interp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
env_config = "config/env.json"
agent_config = "config/agents/DQNAgent/ego_attention_4h.json"

env = load_environment(env_config)
agent = load_agent(agent_config, env)

def load_agent_model(agent, model_path):
    try:
        model_path = agent.load(filename=model_path)
        if model_path:
            print("Loaded {} model from {}".format(agent.__class__.__name__, model_path))
    except NotImplementedError:
        pass

load_agent_model(agent, "../../output/intersection-v0/ego-attention/run_20251221-005400_77723/checkpoint-final.tar")

/Users/vasudharajain/workspace/rl-interp/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/Users/vasudharajain/workspace/rl-interp/.venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment intersection-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(
Preferred device cuda:best unavailable, switching to default cpu


Loaded DQNAgent model from ../../output/intersection-v0/ego-attention/run_20251221-005400_77723/checkpoint-final.tar


In [3]:
agent.value_net

EgoAttentionNetwork(
  (ego_embedding): MultiLayerPerceptron(
    (layers): ModuleList(
      (0): Linear(in_features=7, out_features=64, bias=True)
      (1): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (others_embedding): MultiLayerPerceptron(
    (layers): ModuleList(
      (0): Linear(in_features=7, out_features=64, bias=True)
      (1): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (attention_layer): EgoAttention(
    (value_all): Linear(in_features=64, out_features=64, bias=False)
    (key_all): Linear(in_features=64, out_features=64, bias=False)
    (query_ego): Linear(in_features=64, out_features=64, bias=False)
    (attention_combine): Linear(in_features=64, out_features=64, bias=False)
  )
  (output_layer): MultiLayerPerceptron(
    (layers): ModuleList(
      (0-1): 2 x Linear(in_features=64, out_features=64, bias=True)
    )
    (predict): Linear(in_features=64, out_features=3, bias=True)
  )
)

In [4]:
W_E_ego_0 = agent.value_net.ego_embedding.layers[0].weight.detach().cpu()
W_E_ego_1 = agent.value_net.ego_embedding.layers[1].weight.detach().cpu()
W_E_others_0 = agent.value_net.others_embedding.layers[0].weight.detach().cpu()
W_E_others_1 = agent.value_net.others_embedding.layers[1].weight.detach().cpu()
W_Q = agent.value_net.attention_layer.query_ego.weight.detach().cpu()
W_K = agent.value_net.attention_layer.key_all.weight.detach().cpu()
W_V = agent.value_net.attention_layer.value_all.weight.detach().cpu()
W_O = agent.value_net.attention_layer.attention_combine.weight.detach().cpu()
W_U_out_0 = agent.value_net.output_layer.layers[0].weight.detach().cpu()
W_U_out_1 = agent.value_net.output_layer.layers[1].weight.detach().cpu()
W_U_predict = agent.value_net.output_layer.predict.weight.detach().cpu()

# Combine to get full weight matrices
W_E_ego = W_E_ego_1 @ W_E_ego_0
W_E_others = W_E_others_1 @ W_E_others_0
W_U_out = W_U_out_0 @ W_U_out_1 @ W_U_predict.T
W_OV = W_V @ W_O
W_QK = W_Q @ W_K.T

## Full OV circuit

In [62]:
ov_circuit = W_E_others.T @ W_OV @ W_U_out
fig = px.imshow(
    ov_circuit, 
    color_continuous_scale='RdBu_r', 
    title='Full OV Circuit Weight Matrix', 
    labels={'x':'Output Actions (Ego)', 'y':'Input Features (Others)'},
    x=["SLOWER", "IDLE", "FASTER"],
    y=["presence", "x", "y", "vx", "vy", "cos_h", "sin_h"],
)
fig.update_layout(coloraxis_colorbar=dict(title='Weight Value'))
fig.show()

In [6]:
qk_circuit = W_E_others.T @ W_QK @ W_E_ego
fig = px.imshow(
    qk_circuit, 
    color_continuous_scale='RdBu_r', 
    title='QK Circuit Weight Matrix', 
    labels={'x':'Input Features (Ego)', 'y':'Input Features (Others)'},
    x=["presence", "x", "y", "vx", "vy", "cos_h", "sin_h"],
    y=["presence", "x", "y", "vx", "vy", "cos_h", "sin_h"],
)
fig.update_layout(coloraxis_colorbar=dict(title='Weight Value'))
fig.show()

In [7]:
from collections import defaultdict

class ModelAnalyzer:
    def __init__(self, agent: DQNAgent, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.agent = agent
        self.reset()

    def reset(self):
        state, _ = env.reset()
        self.states = [state]
        self.steps = 0
        self.actions = []
        self.done = False
        self.attention_matrix = []
        self.total_reward = 0

    def run(self):
        self.reset()
        activations = defaultdict(list)

        self.agent.eval()
        def get_activation(name):
            def hook(module, input, output):
                # Handle tuple outputs (e.g., from LSTM, GRU, etc.)
                if isinstance(output, tuple):
                    # Detach all elements in the tuple that are tensors
                    detached_output = tuple(o.detach() if hasattr(o, 'detach') else o for o in output)
                    activations[name].append(detached_output)
                else:
                    activations[name].append(output.detach())
            
            return hook
    
        hooks = []
        for name, sub_module in self.agent.value_net.named_modules():
            if isinstance(sub_module, t.nn.Linear):
                hooks.append(sub_module.register_forward_hook(get_activation(name)))

        while not self.done:
            self.step()
        
        for hook in hooks:
            hook.remove()

        return activations, self.stats()

    def step(self):
        action = self.agent.act(self.states[-1])
        self.actions.append(action)
        state, reward, done, truncated, _ = env.step(action)
        self.total_reward += reward
        self.done = done or truncated
        self.states.append(state)

    def stats(self):
        return {
            "total_reward": self.total_reward,
            "steps": len(self.states),
            "states": len(self.states),
            "actions": len(self.actions),
            "attention_matrices": len(self.attention_matrix),
        }
            

In [8]:
analyzer = ModelAnalyzer(agent)
activations, stats = analyzer.run()

/Users/vasudharajain/workspace/rl-interp/.venv/lib/python3.12/site-packages/rl_agents/agents/deep_q_network/pytorch.py:80: UserWarning:

Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)



In [61]:
Q = t.stack(activations["attention_layer.query_ego"]).mean(dim=0).squeeze(0)  # Shape: (batch, entities, features)
K = t.stack(activations["attention_layer.key_all"]).mean(dim=0).squeeze(0)  # Shape: (batch, entities, features)
Q = Q.repeat(K.shape[0], 1)

attn_pattern = t.einsum('if,jf->ij', Q, K) / t.sqrt(t.tensor(K.shape[-1], dtype=t.float32))
attn_pattern = t.softmax(attn_pattern, dim=-1)  # Shape: (batch, entities, entities)

px.imshow(
    attn_pattern.numpy(), 
    title='Attention Pattern', 
    labels={'x':'Key Entities', 'y':'Query Entities'},
    x=[f'Entity {i}' for i in range(attn_pattern.shape[0])],
    y=[f'Entity {i}' for i in range(attn_pattern.shape[0])],
    color_continuous_scale='RdBu_r'
)

In [15]:
states = t.tensor(analyzer.states[0])
states.shape

torch.Size([15, 7])